In [ ]:
import joblib
import pandas as pd
import numpy as np
import mlflow
import dagshub
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from catboost import CatBoostRegressor
import warnings
warnings.filterwarnings('ignore')
import os

In [11]:
CONFIG = {
    "data_path": "C:\\ESG\\data\\processed_esg_dataset.csv",
    "test_size": 0.25,
    "mlflow_tracing_uri": "https://dagshub.com/virajdeshmukh080818/ESG.mlflow",
    "dagshub_repo_owner": "virajdeshmukh080818",
    "dagshub_repo_name": "ESG",
    "experiment_name": "Model with Hyperparameter"
}

In [12]:
mlflow.set_tracking_uri(CONFIG['mlflow_tracing_uri'])
dagshub.init(repo_owner=CONFIG['dagshub_repo_owner'], repo_name=CONFIG['dagshub_repo_name'], mlflow=True)
mlflow.set_experiment(CONFIG['experiment_name'])

Accessing as virajdeshmukh080818

Initialized MLflow to track repo "virajdeshmukh080818/ESG"

Repository virajdeshmukh080818/ESG initialized!

2025/08/11 18:08:32 INFO mlflow.tracking.fluent: Experiment with name 'Model with Hyperparameter' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/439faf8b781c4c3d8a9b085da017d9cc', creation_time=1754915912837, experiment_id='2', last_update_time=1754915912837, lifecycle_stage='active', name='Model with Hyperparameter', tags={}>

In [2]:
df = pd.read_csv('C:\\ESG\\notebooks\\processed_esg_dataset.csv')

In [3]:
X = df.drop(columns=['MarketCap'])
y = df['MarketCap']

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=CONFIG['test_size'], random_state=42)


In [4]:
model = CatBoostRegressor(verbose=0, random_state=42)
param_grid = {
    'iterations': [200,500,800,1200],
    'depth': [4,6,8,10],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'l2_leaf_reg': [1,3,5,7,9],
    'bagging_temperature': [0,1,6,10]
}

In [13]:
with mlflow.start_run(run_name='Final Model HyperParameter'):
    search= RandomizedSearchCV(
        estimator=model,
        param_distributions=param_grid,
        n_iter=30,
        scoring='r2',
        cv=5,
        verbose=2,
        random_state=42,
        n_jobs=1
    )
    search.fit(X_train, y_train)

    mlflow.log_params(search.best_params_)

    final_model = search.best_estimator_
    final_model.fit(X_train, y_train)

    y_pred = final_model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    mlflow.log_metric('MSE', mse)
    mlflow.log_metric('MAE', mae)
    mlflow.log_metric('RMSE', rmse)
    mlflow.log_metric('R2', r2)

    print("\nFinal Model Performance: ")
    print(f"MSE: {mse:.4f}")
    print(f"MAE: {mae: .4f}")
    print(f"RMSE: {rmse: .4f}")
    print(f"R2: {r2: .4f}")

    model_path = "final_model.pkl"
    joblib.dump(final_model, model_path)
    mlflow.log_artifact(model_path)

    pipeline = Pipeline([('model', final_model)])
    pipeline_path = "final_model_pipeline.pkl"
    joblib.dump(pipeline, pipeline_path)
    mlflow.log_artifact(pipeline_path)

print("Run completed and logged to MLFlow")

Fitting 5 folds for each of 30 candidates, totalling 150 fits
[CV] END bagging_temperature=10, depth=10, iterations=800, l2_leaf_reg=1, learning_rate=0.01; total time= 1.2min
[CV] END bagging_temperature=10, depth=10, iterations=800, l2_leaf_reg=1, learning_rate=0.01; total time= 1.2min
[CV] END bagging_temperature=10, depth=10, iterations=800, l2_leaf_reg=1, learning_rate=0.01; total time= 1.1min
[CV] END bagging_temperature=10, depth=10, iterations=800, l2_leaf_reg=1, learning_rate=0.01; total time= 1.1min
[CV] END bagging_temperature=10, depth=10, iterations=800, l2_leaf_reg=1, learning_rate=0.01; total time= 1.1min
[CV] END bagging_temperature=6, depth=4, iterations=800, l2_leaf_reg=7, learning_rate=0.2; total time=   3.3s
[CV] END bagging_temperature=6, depth=4, iterations=800, l2_leaf_reg=7, learning_rate=0.2; total time=   3.3s
[CV] END bagging_temperature=6, depth=4, iterations=800, l2_leaf_reg=7, learning_rate=0.2; total time=   3.2s
[CV] END bagging_temperature=6, depth=4, it